# Correlation

Two measurements are **correlated** when their uncertainties are not independent: when one reading is too high (it deviates upwards from the true value), the other tends to be too high as well (positive correlation) or too low (negative correlation). This happens whenever the measurements share something, such as

* the same instrument with the same, uncertain calibration, so that every reading is shifted or scaled in the same way,
* the same zero point or reference,
* two parameters of one fit, which compensate each other.

The strength of a correlation is the **correlation coefficient** $\rho = \text{cov}(x, y) / (\sigma_x \sigma_y)$, where $\sigma_x$ and $\sigma_y$ are the standard uncertainties of $x$ and $y$, the numbers after the $\pm$. It is a number between $-1$ and $+1$. For $\rho = 0$ the measurements are independent. With correlations the
first-order propagation becomes

$$
\sigma_f^2 = \sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2
+ 2 \sum_{i < j} \frac{\partial f}{\partial x_i} \frac{\partial f}{\partial x_j}\, \rho_{ij}\, \sigma_i \sigma_j .
$$

For a sum $x + y$ this is $\sigma^2 = \sigma_x^2 + \sigma_y^2 + 2\rho\,\sigma_x\sigma_y$ and for a
difference $x - y$ it is $\sigma^2 = \sigma_x^2 + \sigma_y^2 - 2\rho\,\sigma_x\sigma_y$.

A `Parameter` stores a variance and no covariance, so EasyScience always takes $\rho = 0$. This
notebook shows what that costs and how to do better where it matters.

In [1]:
import numpy as np
import plotly.graph_objects as go

from easyscience import Parameter

The function `show(label, number)` prints a number as `value ± uncertainty unit` and, for a `Parameter` with finite bounds, the bounds. The uncertainty is the attribute `error`; see the note in the arithmetic notebook.

In [2]:
def show(label, number):
    """Print a labelled number as value ± uncertainty with its unit and, for a Parameter, bounds."""
    text = f'{label} = {number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min + 0.0:.4f}, {number.max + 0.0:.4f}]'
    print(text)

In [3]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## A helper that knows about correlation

To propagate correlated uncertainties this notebook uses the helpers `gradient` and `propagate`. They work out
the derivatives numerically and combine them with the formula above. `propagate` takes the function, the
values, the standard uncertainties and, optionally, the matrix of correlation coefficients. Without it the
inputs are independent. `report` prints a result like `show` does.

In [4]:
def gradient(func, values):
    """Derivatives of func with respect to each value, by central differences."""
    values = np.asarray(values, dtype=float)
    slopes = np.empty_like(values)
    for i, value in enumerate(values):
        step = 1e-6 * max(abs(value), 1.0)
        upper, lower = values.copy(), values.copy()
        upper[i] += step
        lower[i] -= step
        slopes[i] = (func(*upper) - func(*lower)) / (2 * step)
    return slopes


def propagate(func, values, sigmas, correlation=None):
    """First-order propagation: the value of func and its uncertainty."""
    sigmas = np.asarray(sigmas, dtype=float)
    if correlation is None:
        correlation = np.eye(len(sigmas))
    covariance = np.asarray(correlation, dtype=float) * np.outer(sigmas, sigmas)
    slopes = gradient(func, values)
    return func(*values), float(np.sqrt(slopes @ covariance @ slopes))


def report(label, result, unit):
    """Print a labelled (value, uncertainty) pair like show does."""
    print(f'{label} = {result[0]:.4f} ± {result[1]:.4f} {unit}')

## The same ruler twice

Two boards are measured one after the other with the same ruler: the first is 1.20 m long and the second 0.80 m. Each reading has a standard uncertainty of ±0.02 m. How strongly the two uncertainties are correlated depends on how much of them comes from the ruler that the readings share, for example an uncertain zero point: if the ruler reads a little long for one board, it does so for the other as well. If all of the uncertainty is shared, $\rho = +1$. First EasyScience's arithmetic, which takes $\rho = 0$, then the helper for different correlations:

In [5]:
first = Parameter(name='first', value=1.20, unit='m', variance=0.02**2)
second = Parameter(name='second', value=0.80, unit='m', variance=0.02**2)

total = first + second
show('total', total)

for rho in [-1.0, 0.0, 0.5, 1.0]:
    result = propagate(lambda a, b: a + b, [1.20, 0.80], [0.02, 0.02], [[1, rho], [rho, 1]])
    print(f'rho = {rho:+.1f}: total = {result[0]:.4f} ± {result[1]:.4f} m')

total = 2.0000 ± 0.0283 m
rho = -1.0: total = 2.0000 ± 0.0000 m
rho = +0.0: total = 2.0000 ± 0.0283 m
rho = +0.5: total = 2.0000 ± 0.0346 m
rho = +1.0: total = 2.0000 ± 0.0400 m


EasyScience's result agrees with $\rho = 0$. If the deviations of the two readings always have the same sign ($\rho = +1$) the uncertainties simply add, $\pm(0.02 + 0.02) = \pm 0.04$ m, which is larger than the $\pm\sqrt{0.02^2 + 0.02^2} = \pm 0.028$ m of independent measurements. If the deviations always had opposite signs ($\rho = -1$) they would cancel completely in the sum.

For a **difference** the roles swap: a deviation that both readings share cancels. The length of the first board
beyond the second, with the same ruler:

In [6]:
difference = first - second
show('difference', difference)

for rho in [-1.0, 0.0, 0.5, 1.0]:
    result = propagate(lambda a, b: a - b, [1.20, 0.80], [0.02, 0.02], [[1, rho], [rho, 1]])
    print(f'rho = {rho:+.1f}: difference = {result[0]:.4f} ± {result[1]:.4f} m')

difference = 0.4000 ± 0.0283 m
rho = -1.0: difference = 0.4000 ± 0.0400 m
rho = +0.0: difference = 0.4000 ± 0.0283 m
rho = +0.5: difference = 0.4000 ± 0.0200 m
rho = +1.0: difference = 0.4000 ± 0.0000 m


The chart shows the uncertainty of the sum and of the difference for every correlation from $-1$ to $+1$. EasyScience's result is the point at $\rho = 0$.

In [7]:
rhos = np.linspace(-1.0, 1.0, 41)
sums, differences = [], []
for rho in rhos:
    correlation = [[1, rho], [rho, 1]]
    sums.append(propagate(lambda a, b: a + b, [1.20, 0.80], [0.02, 0.02], correlation)[1])
    differences.append(propagate(lambda a, b: a - b, [1.20, 0.80], [0.02, 0.02], correlation)[1])

fig = go.Figure()
fig.add_trace(go.Scatter(x=rhos, y=sums, mode='lines', name='sum a + b'))
fig.add_trace(go.Scatter(x=rhos, y=differences, mode='lines', name='difference a - b'))
fig.add_trace(
    go.Scatter(
        x=[0.0],
        y=[total.error],
        mode='markers',
        name='EasyScience (rho = 0)',
        marker=dict(size=10, color='black'),
    )
)
fig.update_layout(
    xaxis_title='correlation coefficient rho',
    yaxis_title='uncertainty / m',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.35, y=0.1),
)
show_figure(fig)

### Exercise 1: Two sections, one ruler

A rod is made of two sections measured with the same tape: 1.20 ± 0.02 m and 0.80 ± 0.03 m. Their uncertainties are strongly correlated, $\rho = 0.9$. What is the total length, with its uncertainty? Compare it with the uncertainty that EasyScience gives.

In [8]:
section_1 = Parameter(name='section_1', value=1.20, unit='m', variance=0.02**2)
section_2 = Parameter(name='section_2', value=0.80, unit='m', variance=0.03**2)

<details class="question">
<summary>Hint 1</summary>

`section_1 + section_2` is what EasyScience computes, with $\rho = 0$. For $\rho = 0.9$ use `propagate` with a $2 \times 2$ matrix of correlation coefficients.

</details>

<details class="question">
<summary>Hint 2</summary>

The matrix is `[[1, 0.9], [0.9, 1]]`: ones on the diagonal and the correlation coefficient off it. The uncertainties are $0.02$ m and $0.03$ m.

</details>

In [9]:
%%solution The correlated uncertainty is 0.049 m, larger than the 0.036 m that EasyScience reports, because a deviation that both readings share adds up in a sum.
by_operators = section_1 + section_2
show('by_operators', by_operators)
report(
    'correlated',
    propagate(lambda a, b: a + b, [1.20, 0.80], [0.02, 0.03], [[1, 0.9], [0.9, 1]]),
    'm',
)

## Shared offsets cancel

The most useful consequence of a positive correlation is in differences: an offset that both readings share
drops out. A surveyor measures the heights of two points from the same station. The station height is uncertain,
which shifts both readings in the same way. The uncertainty of their difference is much smaller than that of either height.

### Exercise 2: A height difference

The heights of two points above a common station are 5.42 ± 0.05 m and 3.18 ± 0.05 m, with a correlation of
$\rho = 0.95$ because of the shared station. What is the height difference, with its uncertainty? What does EasyScience report?

In [10]:
point_a = Parameter(name='point_a', value=5.42, unit='m', variance=0.05**2)
point_b = Parameter(name='point_b', value=3.18, unit='m', variance=0.05**2)

<details class="question">
<summary>Hint 1</summary>

A difference is a function of two values: `lambda a, b: a - b`.

</details>

<details class="question">
<summary>Hint 2</summary>

For $\rho = 0.95$ the variance is $2 \sigma^2 (1 - \rho) = 2 \cdot 0.05^2 \cdot 0.05$.

</details>

In [11]:
%%solution The shared station offset cancels in the difference, so its uncertainty is more than four times smaller than the one EasyScience reports.
by_operators = point_a - point_b
show('by_operators', by_operators)
report(
    'correlated',
    propagate(lambda a, b: a - b, [5.42, 3.18], [0.05, 0.05], [[1, 0.95], [0.95, 1]]),
    'm',
)

## Ratios of readings from one instrument

If two intensities are read with the same detector, an uncertain gain makes both too large or both too small
by the same fraction. In a **ratio** that factor cancels, again because the correlation is positive and the
derivatives have opposite signs.

### Exercise 3: An intensity ratio

Two intensities, 120.0 ± 6.0 and 80.0 ± 4.0 counts per second, were read with the same detector and are
correlated with $\rho = 0.9$. What is their ratio $I_1 / I_2$, with its uncertainty? Compare it with the result of EasyScience.

In [12]:
first_intensity = Parameter(name='first_intensity', value=120.0, unit='1/s', variance=6.0**2)
second_intensity = Parameter(name='second_intensity', value=80.0, unit='1/s', variance=4.0**2)

<details class="question">
<summary>Hint 1</summary>

The ratio is `lambda a, b: a / b`. EasyScience divides the two `Parameter`s directly.

</details>

<details class="question">
<summary>Hint 2</summary>

Both intensities have a relative uncertainty of 5 %. For $\rho = 0.9$ the relative uncertainty of the ratio is $\sqrt{0.05^2 + 0.05^2 - 2 \cdot 0.9 \cdot 0.05 \cdot 0.05}$.

</details>

In [13]:
%%solution The shared gain cancels in the ratio, so the correlated uncertainty is about three times smaller than the one EasyScience reports.
by_operators = first_intensity / second_intensity
show('by_operators', by_operators)
report(
    'correlated',
    propagate(lambda a, b: a / b, [120.0, 80.0], [6.0, 4.0], [[1, 0.9], [0.9, 1]]),
    'dimensionless',
)

## Correlation from a fit

The best-known source of correlated numbers is a fit. The slope and the intercept of a straight line,
$y = a x + b$, are strongly correlated whenever the $x$ values are far from zero: a steeper line must start
lower to pass through the same data. A fit program reports their covariance matrix, and `np.polyfit` can
return it too. These ten points lie close to a line:

In [14]:
x_data = np.arange(10, 20)
y_data = np.array([7.1, 7.4, 8.2, 8.6, 9.3, 9.4, 10.3, 10.6, 11.4, 11.6])

(slope, intercept), covariance = np.polyfit(x_data, y_data, 1, cov=True)
slope_error, intercept_error = np.sqrt(np.diag(covariance))
correlation = covariance[0, 1] / (slope_error * intercept_error)

print(f'slope       = {slope:.4f} ± {slope_error:.4f}')
print(f'intercept   = {intercept:.4f} ± {intercept_error:.4f}')
print(f'correlation = {correlation:.3f}')

slope       = 0.5194 ± 0.0180
intercept   = 1.8588 ± 0.2655
correlation = -0.981


The correlation is close to $-1$. Now predict $y$ at $x = 20$. A `Parameter` keeps only the variance of
each, so EasyScience treats them as independent; the helper uses the full covariance:

In [15]:
fitted_slope = Parameter(name='fitted_slope', value=slope, variance=slope_error**2)
fitted_intercept = Parameter(name='fitted_intercept', value=intercept, variance=intercept_error**2)

at_20 = fitted_slope * 20 + fitted_intercept
show('at_20', at_20)

report(
    'at_20_correlated',
    propagate(
        lambda a, b: a * 20 + b,
        [slope, intercept],
        [slope_error, intercept_error],
        [[1, correlation], [correlation, 1]],
    ),
    'dimensionless',
)

at_20 = 12.2467 ± 0.4467 dimensionless
at_20_correlated = 12.2467 ± 0.1115 dimensionless


Ignoring the correlation overestimates the uncertainty of the prediction by a factor of four. A steeper slope
comes with a lower intercept, so the two compensate each other at $x = 20$. The chart shows the uncertainty
band of the fitted line with and without the correlation. The correct band is narrowest in the middle of the
data, and the independent one is too wide for every $x$ above zero:

In [16]:
x_line = np.linspace(0.0, 25.0, 126)
line = slope * x_line + intercept
with_correlation = np.sqrt(
    slope_error**2 * x_line**2 + intercept_error**2 + 2 * x_line * covariance[0, 1]
)
without_correlation = np.sqrt(slope_error**2 * x_line**2 + intercept_error**2)

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=x_line, y=line, mode='lines', name='fitted line', line=dict(color='black'))
)
fig.add_trace(
    go.Scatter(
        x=x_line,
        y=line + without_correlation,
        mode='lines',
        name='± uncertainty ignoring correlation',
        line=dict(color='rgb(239, 85, 59)', dash='dash'),
    )
)
fig.add_trace(
    go.Scatter(
        x=x_line,
        y=line - without_correlation,
        mode='lines',
        showlegend=False,
        line=dict(color='rgb(239, 85, 59)', dash='dash'),
    )
)
fig.add_trace(
    go.Scatter(
        x=x_line,
        y=line + with_correlation,
        mode='lines',
        name='± uncertainty with correlation',
        line=dict(color='rgb(99, 110, 250)'),
    )
)
fig.add_trace(
    go.Scatter(
        x=x_line,
        y=line - with_correlation,
        mode='lines',
        showlegend=False,
        line=dict(color='rgb(99, 110, 250)'),
    )
)
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers', name='data'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='y',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.02, y=0.98),
)
show_figure(fig, height=480)

### Exercise 4: Predicting from a fit

A fit gave a slope of 0.50 ± 0.02 and an intercept of 2.0 ± 0.3, with a correlation of $-0.97$. What is the
predicted $y$ at $x = 18$, with its uncertainty? What is the uncertainty if the correlation is ignored?

In [17]:
fit_slope = Parameter(name='fit_slope', value=0.50, variance=0.02**2)
fit_intercept = Parameter(name='fit_intercept', value=2.0, variance=0.3**2)

<details class="question">
<summary>Hint 1</summary>

The prediction is `lambda a, b: a * 18 + b`. EasyScience's `fit_slope * 18 + fit_intercept` takes no correlation into account.

</details>

<details class="question">
<summary>Hint 2</summary>

The matrix of correlation coefficients is `[[1, -0.97], [-0.97, 1]]`.

</details>

In [18]:
%%solution The strong negative correlation makes the uncertainty of the prediction almost five times smaller than the one EasyScience reports.
by_operators = fit_slope * 18 + fit_intercept
show('by_operators', by_operators)
report(
    'correlated',
    propagate(lambda a, b: a * 18 + b, [0.50, 2.0], [0.02, 0.3], [[1, -0.97], [-0.97, 1]]),
    'dimensionless',
)

## Good to know

* A correlation coefficient has to come from somewhere: the covariance matrix of a fit, a calibration, or an
  argument about what the measurements share. It cannot be guessed from the values.
* When the correlation is unknown, $\rho = +1$ and $\rho = -1$ give the largest and the smallest uncertainty
  that a combination can have, so they bracket the answer.
* Correlated quantities often hide inside one analysis. If a result is built from several numbers that
  came from the same fit or the same instrument, ask whether the independence assumption of EasyScience holds.
* A `Parameter` stores a value and a variance and nothing else, so there is no place for the covariance of
  two `Parameter`s. Even numbers that came from the same fit, such as the slope and the intercept above, are
  treated as independent by EasyScience's arithmetic and by the functions of `es.math`. Until that changes, use a helper
  like `propagate` with the covariance matrix of the fit.

The first bullet in numbers, with the fit from above. A `Parameter` has no attribute that could hold a
covariance, and the prediction at $x = 20$ shows what that costs:

In [19]:
covariance_attributes = [
    name for name in dir(fitted_slope) if 'covar' in name.lower() or 'correl' in name.lower()
]
with_covariance = propagate(
    lambda a, b: a * 20 + b,
    [slope, intercept],
    [slope_error, intercept_error],
    [[1, correlation], [correlation, 1]],
)[1]

print(f'attributes of a Parameter that can store a covariance: {covariance_attributes or "none"}')
print(f'uncertainty at x = 20   correct {with_covariance:.4f}   EasyScience {at_20.error:.4f}')

attributes of a Parameter that can store a covariance: none
uncertainty at x = 20   correct 0.1115   EasyScience 0.4467
